# 07. Semantic Search with Sentence Embeddings


## 1. What you will build

- A semantic search system using Sentence Transformers + FAISS.
- A keyword baseline using TF-IDF for comparison.
- Retrieval quality evaluation with Recall@k and MRR@k.


## 2. When to use this in real companies

Use this approach when users search with varied wording or paraphrases and exact keyword matching misses relevant results. Common scenarios are support portals, internal documentation search, and knowledge retrieval assistants.


## 3. Business goal

Improve document retrieval relevance for user queries by comparing semantic and lexical retrieval methods.


## 4. Imports and setup


In [ ]:
from pathlib import Path

import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 5. Load datasets from `data/07_data`


In [ ]:
CORPUS_PATH = Path("../data/07_data/knowledge_base.csv")
QUERIES_PATH = Path("../data/07_data/search_queries.csv")

corpus = pd.read_csv(CORPUS_PATH)
queries = pd.read_csv(QUERIES_PATH)

print(f"Corpus size: {len(corpus)}")
print(f"Queries: {len(queries)}")
corpus.head()

In [ ]:
queries.head()

## 6. Semantic retrieval (SentenceTransformer + FAISS)


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(corpus["text"].tolist(), normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))


def semantic_search(query: str, k: int = 3):
    """Return top-k semantic matches for one query."""
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_emb, k)
    out = corpus.iloc[idx[0]].copy()
    out["score"] = scores[0]
    return out[["doc_id", "text", "score"]].reset_index(drop=True)

semantic_search("How can I reset my account password?")

## 7. Keyword baseline (TF-IDF cosine)


In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(corpus["text"])


def keyword_search(query: str, k: int = 3):
    """Return top-k keyword matches for one query."""
    q_vec = tfidf.transform([query])
    scores = cosine_similarity(q_vec, X)[0]
    idx = scores.argsort()[-k:][::-1]
    out = corpus.iloc[idx].copy()
    out["score"] = scores[idx]
    return out[["doc_id", "text", "score"]].reset_index(drop=True)

keyword_search("How can I reset my account password?")

## 8. Evaluate retrieval quality


In [ ]:
def reciprocal_rank(retrieved_ids, expected_id):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id == expected_id:
            return 1.0 / rank
    return 0.0


rows = []
for _, row in queries.iterrows():
    sem_ids = semantic_search(row["query"], k=3)["doc_id"].tolist()
    key_ids = keyword_search(row["query"], k=3)["doc_id"].tolist()

    rows.append(
        {
            "query": row["query"],
            "expected_doc_id": row["expected_doc_id"],
            "semantic_hit@3": int(row["expected_doc_id"] in sem_ids),
            "semantic_mrr@3": reciprocal_rank(sem_ids, row["expected_doc_id"]),
            "keyword_hit@3": int(row["expected_doc_id"] in key_ids),
            "keyword_mrr@3": reciprocal_rank(key_ids, row["expected_doc_id"]),
        }
    )

eval_df = pd.DataFrame(rows)
eval_df.head()

In [ ]:
summary = pd.DataFrame(
    {
        "method": ["semantic_embedding_faiss", "keyword_tfidf"],
        "recall@3": [eval_df["semantic_hit@3"].mean(), eval_df["keyword_hit@3"].mean()],
        "mrr@3": [eval_df["semantic_mrr@3"].mean(), eval_df["keyword_mrr@3"].mean()],
    }
).round(3)
summary

## 9. Production-style query helper


In [ ]:
def search_knowledge_base(query: str, method: str = "semantic", k: int = 3):
    """Search the KB with the selected method (`semantic` or `keyword`)."""
    if method == "semantic":
        return semantic_search(query, k=k)
    if method == "keyword":
        return keyword_search(query, k=k)
    raise ValueError("method must be 'semantic' or 'keyword'")

search_knowledge_base("Where do I configure retention rules?", method="semantic", k=3)

## 10. Summary

- Data is externalized under `data/07_data` for reproducibility.
- The notebook compares semantic and lexical retrieval with measurable metrics.
- This is a practical pattern for enterprise search and knowledge assistants.
